In [6]:
# ==========================================
# CELL 1: IMPORTS & CONFIGURATION
# ==========================================
import os
import numpy as np
import joblib
import shutil
import datetime
from sklearn.model_selection import train_test_split
from sklearn.ensemble import AdaBoostClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix

# ── DATA PATHS ───────────────────────────────────────────
# First run:    point to train_X.npy / train_y.npy
# Retraining:   run npyfile_combiner.py first, then point to combined files
X_PATH = r"C:\Users\kalig\OneDrive\Desktop\train_X.npy"
Y_PATH = r"C:\Users\kalig\OneDrive\Desktop\train_y.npy"

# ── MODEL PATHS ──────────────────────────────────────────
MODEL_SAVE_PATH   = r"C:\Users\kalig\OneDrive\Desktop\tissue_density_model.joblib"
MODEL_BACKUP_DIR  = r"C:\Users\kalig\OneDrive\Desktop\model_backups"

RANDOM_STATE = 42
os.makedirs(MODEL_BACKUP_DIR, exist_ok=True)

In [7]:
# ==========================================
# CELL 2: BACKUP OLD MODEL BEFORE RETRAINING
# ==========================================
# Skips silently if no model exists yet (first run)
if os.path.exists(MODEL_SAVE_PATH):
    timestamp   = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
    backup_path = os.path.join(MODEL_BACKUP_DIR, f"tissue_density_model_{timestamp}.joblib")
    shutil.copy2(MODEL_SAVE_PATH, backup_path)
    print(f" Old model backed up → {backup_path}")
else:
    print("No existing model found — fresh training run, skipping backup.")

 Old model backed up → C:\Users\kalig\OneDrive\Desktop\model_backups\tissue_density_model_20260610_142817.joblib


In [8]:
# ==========================================
# CELL 3: LOAD & BALANCE FEATURES
# ==========================================
from imblearn.over_sampling import SMOTE
from collections import Counter

print("Loading feature matrix and labels...")
X = np.load(X_PATH)
y = np.load(Y_PATH)
print(f"Data loaded! Shape: X={X.shape}, y={y.shape}")

unique, counts = np.unique(y, return_counts=True)
print(f"Full dataset class distribution: {dict(zip(unique, counts))}")

# Split FIRST — test set must never be touched by SMOTE
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

print(f"\nBefore SMOTE — Train: {Counter(y_train)}")

# Apply SMOTE only to training set
smote = SMOTE(sampling_strategy='auto', random_state=RANDOM_STATE, k_neighbors=7)
X_train_balanced, y_train_balanced = smote.fit_resample(X_train, y_train)

print(f"After  SMOTE — Train: {Counter(y_train_balanced)}")

# Summary
unique_b, counts_b = np.unique(y_train_balanced, return_counts=True)
synthetic = X_train_balanced.shape[0] - X_train.shape[0]

print(f"\n{'='*45}")
print(f"  Original training images  : {X_train.shape[0]}")
print(f"  Synthetic images added    : {synthetic}")
print(f"  Total training images     : {X_train_balanced.shape[0]}")
print(f"  Test images (untouched)   : {X_test.shape[0]}")
print(f"  Features per image        : {X_train_balanced.shape[1]}")
print(f"{'='*45}")

Loading feature matrix and labels...
Data loaded! Shape: X=(9999, 1792), y=(9999,)
Full dataset class distribution: {np.int64(1): np.int64(1172), np.int64(2): np.int64(4282), np.int64(3): np.int64(3991), np.int64(4): np.int64(554)}

Before SMOTE — Train: Counter({np.int64(2): 3425, np.int64(3): 3193, np.int64(1): 938, np.int64(4): 443})
After  SMOTE — Train: Counter({np.int64(1): 3425, np.int64(2): 3425, np.int64(3): 3425, np.int64(4): 3425})

  Original training images  : 7999
  Synthetic images added    : 5701
  Total training images     : 13700
  Test images (untouched)   : 2000
  Features per image        : 1792


In [9]:
# ==========================================
# CELL 4: STRATEGY C (ORDINAL ADABOOST)
# ==========================================
class OrdinalAdaBoost:
    """
    Strategy C: Ordinal Classification Ensemble.
    Trains K-1 binary AdaBoost classifiers on ordinal thresholds.
    """
    def __init__(self, n_estimators=50, random_state=None):
        self.n_estimators = n_estimators
        self.random_state = random_state
        self.models       = []
        self.classes_     = None

    def fit(self, X: np.ndarray, y: np.ndarray) -> None:
        self.classes_ = np.sort(np.unique(y))
        K = len(self.classes_)

        if K < 3:
            raise ValueError(f"OrdinalAdaBoost requires at least 3 ordered classes, got {K}.")

        self.models = []

        for i in range(K - 1):
            threshold = self.classes_[i]
            y_binary  = (y > threshold).astype(int)

            model = AdaBoostClassifier(
                estimator=DecisionTreeClassifier(max_depth=1),
                n_estimators=self.n_estimators,
                random_state=self.random_state
            )
            model.fit(X, y_binary)
            self.models.append(model)
            print(f"   Binary classifier {i+1}/{K-1} trained "
                  f"(threshold: class {threshold} vs rest)")

    def predict(self, X: np.ndarray) -> np.ndarray:
        votes       = np.column_stack([m.predict(X) for m in self.models])
        ordinal_idx = np.clip(votes.sum(axis=1).astype(int), 0, len(self.classes_) - 1)
        return self.classes_[ordinal_idx]

In [10]:
# ==========================================
# CELL 5: TRAIN, EVALUATE & SAVE
# ==========================================
from sklearn.metrics import classification_report

print("Training Ordinal AdaBoost (Strategy C)...")
model = OrdinalAdaBoost(n_estimators=150, random_state=RANDOM_STATE)
model.fit(X_train_balanced, y_train_balanced)

print("\nEvaluating on test set...")
y_pred = model.predict(X_test)

acc = accuracy_score(y_test, y_pred)
cm = confusion_matrix(y_test, y_pred)

print(f"\n{'='*45}")
print(f" PERFORMANCE REPORT")
print(f"{'='*45}")
print(f" Images model trained on : {X_train_balanced.shape[0]}")
print(f" ↳ Real images          : {X_train.shape[0]}")
print(f" ↳ SMOTE synthetic      : {X_train_balanced.shape[0] - X_train.shape[0]}")
print(f" Images tested on       : {X_test.shape[0]}")
print(f" Overall Accuracy       : {acc:.4f}")
print(f"{'='*45}")

print("\nPer-Class Breakdown:")
print(classification_report(
    y_test,
    y_pred,
    target_names=[
        'Tissue Density Category 1',
        'Tissue Density Category 2',
        'Tissue Density Category 3',
        'Tissue Density Category 4'
    ]
))

print("Confusion Matrix:")
header = "              " + "  ".join(f"Pred {i+1}" for i in range(4))
print(header)
for i, row in enumerate(cm):
    print(f"Actual {i+1}     " + "  ".join(f"{v:6d}" for v in row))

print(f"\nSaving model to {MODEL_SAVE_PATH}...")
joblib.dump(model, MODEL_SAVE_PATH)
print(" Done! Balanced Ordinal Model saved.")

Training Ordinal AdaBoost (Strategy C)...
   Binary classifier 1/3 trained (threshold: class 1 vs rest)
   Binary classifier 2/3 trained (threshold: class 2 vs rest)
   Binary classifier 3/3 trained (threshold: class 3 vs rest)

Evaluating on test set...

 PERFORMANCE REPORT
 Images model trained on : 13700
 ↳ Real images          : 7999
 ↳ SMOTE synthetic      : 5701
 Images tested on       : 2000
 Overall Accuracy       : 0.6685

Per-Class Breakdown:
                           precision    recall  f1-score   support

Tissue Density Category 1       0.54      0.71      0.61       234
Tissue Density Category 2       0.73      0.66      0.69       857
Tissue Density Category 3       0.74      0.66      0.70       798
Tissue Density Category 4       0.37      0.66      0.47       111

                 accuracy                           0.67      2000
                macro avg       0.59      0.67      0.62      2000
             weighted avg       0.69      0.67      0.67      2000

Conf